# BL-113 Spike — Binary Preprocessing Validation

Compare **markitdown** (Python, Microsoft) vs **npm packages** (pdf-parse, mammoth) for
converting PDF/DOCX/PPTX to plain text before LLM processing.

Test dataset: real TIMining files from QA worktree.

In [1]:
# Cell 1: Setup
import subprocess, os, time

QA_SOURCES = '/Users/nlundin/Dev/repos/pd-spec--qa/01_Sources'
VENV_PYTHON = '/tmp/markitdown-venv/bin/python3.13'
APP_DIR = '/Users/nlundin/Dev/repos/pd-spec/app'

FILES = {
    'pdf_slides': f'{QA_SOURCES}/PPT TIMining - General_ENE 2026_Español.pdf',
    'pptx_diagram': f'{QA_SOURCES}/TIMining_8-Layer_Diagram.pptx',
    'docx_narrative': f'{QA_SOURCES}/Narrativa CCB para Mngt Team_ESP.docx',
    'docx_usecases': f'{QA_SOURCES}/Workshop 1/Casos de Uso Workshop 01.docx',
    'docx_retro': f'{QA_SOURCES}/Workshop 1/KEEP-STOP-START (retro).docx',
}

# Verify all files exist
for key, path in FILES.items():
    size = os.path.getsize(path)
    print(f'  {key}: {size:,} bytes — {os.path.basename(path)}')

def preview(text, max_chars=500):
    """Show truncated preview with char count."""
    clean = text.strip()
    chars = len(clean)
    lines = clean.count('\n') + 1
    print(f'  [{chars:,} chars, {lines} lines]')
    if chars > max_chars:
        print(clean[:max_chars] + '\n  [...truncated]')
    else:
        print(clean)
    return chars

# Store results for comparison
results = {}

  pdf_slides: 11,970,209 bytes — PPT TIMining - General_ENE 2026_Español.pdf
  pptx_diagram: 56,769 bytes — TIMining_8-Layer_Diagram.pptx
  docx_narrative: 26,777 bytes — Narrativa CCB para Mngt Team_ESP.docx
  docx_usecases: 30,059 bytes — Casos de Uso Workshop 01.docx
  docx_retro: 11,401 bytes — KEEP-STOP-START (retro).docx


## Cell 2: markitdown — all 5 files

Uses `python3.13 -m markitdown <file>` via subprocess.

In [2]:
# Cell 2: markitdown extraction
print('=' * 60)
print('APPROACH A: markitdown (Python, Microsoft)')
print('=' * 60)

markitdown_results = {}

for key, path in FILES.items():
    print(f'\n--- {key} ---')
    print(f'  File: {os.path.basename(path)}')
    
    start = time.time()
    try:
        result = subprocess.run(
            [VENV_PYTHON, '-m', 'markitdown', path],
            capture_output=True, text=True, timeout=60
        )
        elapsed = time.time() - start
        
        if result.returncode != 0:
            print(f'  ERROR (exit {result.returncode}): {result.stderr[:200]}')
            markitdown_results[key] = {'chars': 0, 'text': '', 'time': elapsed, 'error': result.stderr[:200]}
        else:
            text = result.stdout
            chars = preview(text)
            print(f'  Time: {elapsed:.2f}s')
            markitdown_results[key] = {'chars': chars, 'text': text, 'time': elapsed, 'error': None}
    except subprocess.TimeoutExpired:
        print(f'  TIMEOUT (>60s)')
        markitdown_results[key] = {'chars': 0, 'text': '', 'time': 60, 'error': 'TIMEOUT'}

results['markitdown'] = markitdown_results

APPROACH A: markitdown (Python, Microsoft)

--- pdf_slides ---
  File: PPT TIMining - General_ENE 2026_Español.pdf


  [31,310 chars, 2270 lines]
TIMining

Presentación de la compañía, sus soluciones y
valor en la minería de rajo abierto

Diciembre 2025

Resumen ejecutivo

• TIMining es una empresa 100% dedicada al desarrollo tecnológico para la minería, con cerca de 15 años de experiencia en

la industria, presencia en más de 50 minas en 8 países y que se destaca por desarrollar soluciones para mejora de la
gestión y toma de decisión en la planificación y operación minera de rajo abierto.

• Nuestro foco está en ayudar a solucionar una 
  [...truncated]
  Time: 2.98s

--- pptx_diagram ---
  File: TIMining_8-Layer_Diagram.pptx


  [1,305 chars, 22 lines]
<!-- Slide number: 1 -->
TIMINING Platform – 8‑Layer Interconnection Diagram
Prescriptions & control
Upward data flow

8) AI Copilot & Autonomy Enablement
Project TIM natural‑language diagnostics, prescriptive actions; autonomy‑ready geometry feeds to robots/dispatch

7) Analytics, Anomalies & Auto‑Projections
Soft‑sensor monitoring, performance‑to‑plan KPIs, automatic short‑horizon simulations with Orchestra
6) Situational Awareness & UX
Role‑specific real‑time views & dashboards (Aware): super
  [...truncated]
  Time: 0.60s

--- docx_narrative ---
  File: Narrativa CCB para Mngt Team_ESP.docx


  [14,370 chars, 114 lines]
## El Eslabón Perdido de la Minería Autónoma: La Verdad Geométrica en Tiempo Real

#### Por Carlo Calderón, CTO & CPO, Timining

### La Tesis Fundamental: La Geometría es el Negocio

En la minería a, el plan de negocio no es una hoja de cálculo financiera; es un volumen geométrico que debe moverse en un tiempo específico. Sin embargo, históricamente, la industria ha operado bajo una paradoja costosa: la Planificación vive en un mundo estático y perfecto (el diseño), mientras que la Operación viv
  [...truncated]
  Time: 0.66s

--- docx_usecases ---
  File: Casos de Uso Workshop 01.docx


  [15,296 chars, 653 lines]
**LAMINAS DE CASO USO**

**ANEXOS WORKSHOP**

Aquí está el documento con el formato de cada lámina ajustado para seguir la estructura clara y consistente de la Lámina 1, corrigiendo el orden, el uso de negritas y los saltos de línea.

-----

### Lámina 1: Gerente Minas (Cierre Mensual)

**¿Quién es?**

* Gerente Minas.

**¿Dónde ocurre?**

* En la reunión del cierre mensual.

**Situación de estrés / Escenario de crisis**

* Se cumple el mineral y el lastre pero la Adherencia / recuperación de la
  [...truncated]
  Time: 1.39s

--- docx_retro ---
  File: KEEP-STOP-START (retro).docx


  [5,355 chars, 106 lines]
# **Lo que NO necesita el usuario**

*(no agrega valor)*

* LA Visualización 3D de La Mina
* Dejar de hacer desarrollos específicos y desarrollarlos de manera genérica
* Gráficos o reportes que nos piden porque a ellos le piden SIN SOLUCIONAR ALGO
* MOSTRAR LOS MISMOS KPIS QUE TODOS LOS DASHBOARDS
* Repetir Información -> Generar confusión
* Gastar/perder tiempo
* Capacitaciones muy largas
* No quiere más problemas (TI, contrato, adherencia) quiere que nuestra solución sea plug and play
* Una ex
  [...truncated]
  Time: 0.68s


## Cell 3: npm — PDF (pdf-parse)

In [3]:
# Cell 3: pdf-parse (Node.js)
print('=' * 60)
print('APPROACH B: pdf-parse (Node.js)')
print('=' * 60)

npm_results = {}

pdf_path = FILES['pdf_slides']
print(f'\n--- pdf_slides ---')
print(f'  File: {os.path.basename(pdf_path)}')

node_script = f"""
const fs = require('fs');
const pdfParse = require('pdf-parse');
const buf = fs.readFileSync({repr(pdf_path)});
pdfParse(buf).then(data => {{
    process.stdout.write(data.text);
}}).catch(err => {{
    process.stderr.write(err.message);
    process.exit(1);
}});
"""

start = time.time()
try:
    result = subprocess.run(
        ['node', '-e', node_script],
        capture_output=True, text=True, timeout=60,
        cwd=APP_DIR
    )
    elapsed = time.time() - start
    
    if result.returncode != 0:
        print(f'  ERROR: {result.stderr[:200]}')
        npm_results['pdf_slides'] = {'chars': 0, 'text': '', 'time': elapsed, 'error': result.stderr[:200]}
    else:
        text = result.stdout
        chars = preview(text)
        print(f'  Time: {elapsed:.2f}s')
        npm_results['pdf_slides'] = {'chars': chars, 'text': text, 'time': elapsed, 'error': None}
except subprocess.TimeoutExpired:
    print(f'  TIMEOUT (>60s)')
    npm_results['pdf_slides'] = {'chars': 0, 'text': '', 'time': 60, 'error': 'TIMEOUT'}

APPROACH B: pdf-parse (Node.js)

--- pdf_slides ---
  File: PPT TIMining - General_ENE 2026_Español.pdf


  ERROR: [eval]:5
pdfParse(buf).then(data => {
^

TypeError: pdfParse is not a function
    at [eval]:5:1
    at runScriptInThisContext (node:internal/vm:219:10)
    at node:internal/process/exe


## Cell 4: npm — DOCX (mammoth)

In [4]:
# Cell 4: mammoth for DOCX files (Node.js)
print('=' * 60)
print('APPROACH B: mammoth (Node.js) — DOCX files')
print('=' * 60)

docx_keys = ['docx_narrative', 'docx_usecases', 'docx_retro']

for key in docx_keys:
    path = FILES[key]
    print(f'\n--- {key} ---')
    print(f'  File: {os.path.basename(path)}')
    
    # mammoth extracts to raw text (no HTML)
    node_script = f"""
const mammoth = require('mammoth');
mammoth.extractRawText({{ path: {repr(path)} }})
    .then(result => process.stdout.write(result.value))
    .catch(err => {{ process.stderr.write(err.message); process.exit(1); }});
"""
    
    start = time.time()
    try:
        result = subprocess.run(
            ['node', '-e', node_script],
            capture_output=True, text=True, timeout=60,
            cwd=APP_DIR
        )
        elapsed = time.time() - start
        
        if result.returncode != 0:
            print(f'  ERROR: {result.stderr[:200]}')
            npm_results[key] = {'chars': 0, 'text': '', 'time': elapsed, 'error': result.stderr[:200]}
        else:
            text = result.stdout
            chars = preview(text)
            print(f'  Time: {elapsed:.2f}s')
            npm_results[key] = {'chars': chars, 'text': text, 'time': elapsed, 'error': None}
    except subprocess.TimeoutExpired:
        print(f'  TIMEOUT (>60s)')
        npm_results[key] = {'chars': 0, 'text': '', 'time': 60, 'error': 'TIMEOUT'}

APPROACH B: mammoth (Node.js) — DOCX files

--- docx_narrative ---
  File: Narrativa CCB para Mngt Team_ESP.docx
  [14,098 chars, 125 lines]
El Eslabón Perdido de la Minería Autónoma: La Verdad Geométrica en Tiempo Real

Por Carlo Calderón, CTO & CPO, Timining

La Tesis Fundamental: La Geometría es el Negocio

En la minería a, el plan de negocio no es una hoja de cálculo financiera; es un volumen geométrico que debe moverse en un tiempo específico. Sin embargo, históricamente, la industria ha operado bajo una paradoja costosa: la Planificación vive en un mundo estático y perfecto (el diseño), mientras que la Operación vive en un caos
  [...truncated]
  Time: 0.12s

--- docx_usecases ---
  File: Casos de Uso Workshop 01.docx


  [14,468 chars, 837 lines]
LAMINAS DE CASO USO



ANEXOS WORKSHOP



Aquí está el documento con el formato de cada lámina ajustado para seguir la estructura clara y consistente de la Lámina 1, corrigiendo el orden, el uso de negritas y los saltos de línea.



-----

Lámina 1: Gerente Minas (Cierre Mensual)



¿Quién es?

Gerente Minas.

¿Dónde ocurre?

En la reunión del cierre mensual.

Situación de estrés / Escenario de crisis

Se cumple el mineral y el lastre pero la Adherencia / recuperación de la planta es mala (100%)
  [...truncated]
  Time: 0.20s

--- docx_retro ---
  File: KEEP-STOP-START (retro).docx
  [5,245 chars, 207 lines]
Lo que NO necesita el usuario

(no agrega valor)

LA Visualización 3D de La Mina

Dejar de hacer desarrollos específicos y desarrollarlos de manera genérica

Gráficos o reportes que nos piden porque a ellos le piden SIN SOLUCIONAR ALGO

MOSTRAR LOS MISMOS KPIS QUE TODOS LOS DASHBOARDS

Repetir Información -> Generar confusión

Gastar/perder tiempo

Capac

## Cell 5: npm — PPTX

No strong npm package for PPTX text extraction. Try `pptx-parser` if installed,
otherwise markitdown wins this format by default.

In [5]:
# Cell 5: PPTX via npm — check if any package available
print('=' * 60)
print('APPROACH B: PPTX via npm')
print('=' * 60)

pptx_path = FILES['pptx_diagram']
print(f'\nFile: {os.path.basename(pptx_path)}')

# Try basic ZIP + XML extraction (PPTX is a ZIP of XMLs)
node_script = f"""
const fs = require('fs');
const {{ execSync }} = require('child_process');

// PPTX is a ZIP — extract slide XMLs and strip tags
const path = {repr(pptx_path)};
const tmpDir = '/tmp/pptx-extract-' + Date.now();
try {{
    execSync(`mkdir -p ${{tmpDir}} && unzip -o "${{path}}" 'ppt/slides/*.xml' -d ${{tmpDir}}`, {{ stdio: 'pipe' }});
    const slidesDir = tmpDir + '/ppt/slides';
    if (fs.existsSync(slidesDir)) {{
        const files = fs.readdirSync(slidesDir).filter(f => f.endsWith('.xml')).sort();
        let allText = '';
        for (const f of files) {{
            const xml = fs.readFileSync(slidesDir + '/' + f, 'utf8');
            // Extract text between <a:t> tags
            const texts = [];
            xml.replace(/<a:t>([^<]*)<\/a:t>/g, (_, t) => texts.push(t));
            if (texts.length) {{
                allText += '--- ' + f + ' ---\n' + texts.join(' ') + '\n\n';
            }}
        }}
        process.stdout.write(allText || '(no text found in slides)');
    }} else {{
        process.stdout.write('(no slides directory found)');
    }}
    execSync(`rm -rf ${{tmpDir}}`);
}} catch(e) {{
    process.stderr.write(e.message);
    process.exit(1);
}}
"""

start = time.time()
try:
    result = subprocess.run(
        ['node', '-e', node_script],
        capture_output=True, text=True, timeout=60,
        cwd=APP_DIR
    )
    elapsed = time.time() - start
    
    if result.returncode != 0:
        print(f'  ERROR: {result.stderr[:200]}')
        npm_results['pptx_diagram'] = {'chars': 0, 'text': '', 'time': elapsed, 'error': result.stderr[:200]}
    else:
        text = result.stdout
        chars = preview(text)
        print(f'  Time: {elapsed:.2f}s')
        npm_results['pptx_diagram'] = {'chars': chars, 'text': text, 'time': elapsed, 'error': None}
except subprocess.TimeoutExpired:
    print(f'  TIMEOUT (>60s)')
    npm_results['pptx_diagram'] = {'chars': 0, 'text': '', 'time': 60, 'error': 'TIMEOUT'}

results['npm'] = npm_results

APPROACH B: PPTX via npm

File: TIMining_8-Layer_Diagram.pptx
  ERROR: [eval]:20
                allText += '--- ' + f + ' ---
                                        ^^^^^
Expected ';', '}' or <eof>

SyntaxError: Invalid or unexpected token
    at makeContextifyScr


<>:45: SyntaxWarning: invalid escape sequence '\/'
<>:45: SyntaxWarning: invalid escape sequence '\/'
/var/folders/3p/v5ghn39d159g4dkgbht1mmfc0000gn/T/ipykernel_32821/2248309068.py:45: SyntaxWarning: invalid escape sequence '\/'
  result = subprocess.run(


## Cell 6: Comparison Table

In [6]:
# Cell 6: Side-by-side comparison
print('=' * 80)
print('COMPARISON TABLE')
print('=' * 80)

mk = results.get('markitdown', {})
npm = results.get('npm', {})

header = f'{"File":<20} {"Format":<6} {"markitdown":>12} {"npm":>12} {"MK time":>8} {"npm time":>8} {"Winner":<12}'
print(header)
print('-' * len(header))

format_map = {
    'pdf_slides': 'PDF',
    'pptx_diagram': 'PPTX',
    'docx_narrative': 'DOCX',
    'docx_usecases': 'DOCX',
    'docx_retro': 'DOCX',
}

for key in FILES:
    fmt = format_map[key]
    mk_data = mk.get(key, {})
    npm_data = npm.get(key, {})
    
    mk_chars = mk_data.get('chars', 0)
    npm_chars = npm_data.get('chars', 0)
    mk_time = mk_data.get('time', 0)
    npm_time = npm_data.get('time', 0)
    mk_err = mk_data.get('error')
    npm_err = npm_data.get('error')
    
    # Determine winner
    if mk_err and npm_err:
        winner = 'BOTH FAIL'
    elif mk_err:
        winner = 'npm'
    elif npm_err:
        winner = 'markitdown'
    elif mk_chars > npm_chars * 1.2:
        winner = 'markitdown'
    elif npm_chars > mk_chars * 1.2:
        winner = 'npm'
    else:
        winner = 'TIE'
    
    mk_str = f'{mk_chars:,}' if not mk_err else f'ERR'
    npm_str = f'{npm_chars:,}' if not npm_err else f'N/A'
    
    print(f'{key:<20} {fmt:<6} {mk_str:>12} {npm_str:>12} {mk_time:>7.2f}s {npm_time:>7.2f}s {winner:<12}')

print()
print('Winner criteria: >20% more chars extracted = winner. Otherwise TIE.')
print('N/A = no npm package tested for this format.')

COMPARISON TABLE
File                 Format   markitdown          npm  MK time npm time Winner      
------------------------------------------------------------------------------------
pdf_slides           PDF          31,310          N/A    2.98s    0.25s markitdown  
pptx_diagram         PPTX          1,305          N/A    0.60s    0.06s markitdown  
docx_narrative       DOCX         14,370       14,098    0.66s    0.12s TIE         
docx_usecases        DOCX         15,296       14,468    1.39s    0.20s TIE         
docx_retro           DOCX          5,355        5,245    0.68s    0.09s TIE         

Winner criteria: >20% more chars extracted = winner. Otherwise TIE.
N/A = no npm package tested for this format.


## Cell 7: Vision Detection Heuristic

In [7]:
# Cell 7: Vision detection heuristic validation
print('=' * 80)
print('VISION DETECTION HEURISTIC')
print('=' * 80)
print()
print('Rule: needs_vision = (extracted_chars < 200) AND (file_size > 100KB)')
print()

# Use markitdown results as the primary extractor
mk = results.get('markitdown', {})

header = f'{"File":<20} {"Size":>10} {"Chars":>8} {"needs_vision":<14} {"Expected":<14} {"Match":<5}'
print(header)
print('-' * len(header))

expected_vision = {
    'pdf_slides': True,       # 11MB exported slides — mostly visual
    'pptx_diagram': True,     # Diagram — mostly visual
    'docx_narrative': False,  # Narrative text
    'docx_usecases': False,   # Use cases — structured text
    'docx_retro': False,      # Retro notes
}

for key, path in FILES.items():
    file_size = os.path.getsize(path)
    mk_data = mk.get(key, {})
    chars = mk_data.get('chars', 0)
    
    needs_vision = chars < 200 and file_size > 100_000
    expected = expected_vision[key]
    match = '✓' if needs_vision == expected else '✗'
    
    size_str = f'{file_size:,}'
    print(f'{key:<20} {size_str:>10} {chars:>8} {str(needs_vision):<14} {str(expected):<14} {match:<5}')

print()
print('NOTE: If heuristic misses visual files that DO have text layers,')
print('the text extraction still works — just the content may be sparse/unlabeled.')
print('Heuristic is a SAFETY NET for files where extraction produces near-zero text.')

VISION DETECTION HEURISTIC

Rule: needs_vision = (extracted_chars < 200) AND (file_size > 100KB)

File                       Size    Chars needs_vision   Expected       Match
----------------------------------------------------------------------------
pdf_slides           11,970,209    31310 False          True           ✗    
pptx_diagram             56,769     1305 False          True           ✗    
docx_narrative           26,777    14370 False          False          ✓    
docx_usecases            30,059    15296 False          False          ✓    
docx_retro               11,401     5355 False          False          ✓    

NOTE: If heuristic misses visual files that DO have text layers,
the text extraction still works — just the content may be sparse/unlabeled.
Heuristic is a SAFETY NET for files where extraction produces near-zero text.


## Cell 8: Quality Assessment + Content Samples

Deeper look at extracted content for quality scoring.

In [8]:
# Cell 8: Quality assessment — check Spanish chars, structure preservation
print('=' * 80)
print('QUALITY ASSESSMENT')
print('=' * 80)

spanish_chars = set('áéíóúñüÁÉÍÓÚÑÜ¿¡')

mk = results.get('markitdown', {})
npm = results.get('npm', {})

for key in FILES:
    print(f'\n{"=" * 40}')
    print(f'FILE: {key}')
    print(f'{"=" * 40}')
    
    for approach, data in [('markitdown', mk.get(key, {})), ('npm', npm.get(key, {}))]:
        text = data.get('text', '')
        if not text:
            print(f'  {approach}: (no output)')
            continue
        
        # Spanish character check
        found_spanish = [c for c in text if c in spanish_chars]
        has_spanish = len(found_spanish) > 0
        unique_spanish = set(found_spanish)
        
        # Structure check (headings, bullets, tables)
        has_headings = text.count('#') > 0
        has_bullets = text.count('- ') > 2 or text.count('• ') > 2
        has_tables = '|' in text and '-|-' in text.replace(' ', '')
        
        print(f'  {approach}:')
        print(f'    Chars: {len(text.strip()):,}')
        print(f'    Spanish chars: {"YES" if has_spanish else "NO"} ({len(unique_spanish)} unique: {"".join(sorted(unique_spanish))})')
        print(f'    Headings: {"YES" if has_headings else "NO"}')
        print(f'    Bullets/lists: {"YES" if has_bullets else "NO"}')
        print(f'    Tables: {"YES" if has_tables else "NO"}')
        # First 300 chars sample
        print(f'    Sample: {text.strip()[:300]}...')

QUALITY ASSESSMENT

FILE: pdf_slides
  markitdown:
    Chars: 31,310
    Spanish chars: YES (8 unique: ¿Ááéíñóú)
    Headings: NO
    Bullets/lists: YES
    Tables: NO
    Sample: TIMining

Presentación de la compañía, sus soluciones y
valor en la minería de rajo abierto

Diciembre 2025

Resumen ejecutivo

• TIMining es una empresa 100% dedicada al desarrollo tecnológico para la minería, con cerca de 15 años de experiencia en

la industria, presencia en más de 50 minas en 8 ...
  npm: (no output)

FILE: pptx_diagram
  markitdown:
    Chars: 1,305
    Spanish chars: NO (0 unique: )
    Headings: NO
    Bullets/lists: NO
    Tables: NO
    Sample: <!-- Slide number: 1 -->
TIMINING Platform – 8‑Layer Interconnection Diagram
Prescriptions & control
Upward data flow

8) AI Copilot & Autonomy Enablement
Project TIM natural‑language diagnostics, prescriptive actions; autonomy‑ready geometry feeds to robots/dispatch

7) Analytics, Anomalies & Auto‑...
  npm: (no output)

FILE: docx_narrative


## Cell 9: Final Decision Matrix

In [9]:
# Cell 9: Decision matrix summary
print('=' * 80)
print('DECISION MATRIX — Fill after reviewing results above')
print('=' * 80)
print()
print('| Format | markitdown | npm         | Winner | Notes |')
print('|--------|-----------|-------------|--------|-------|')

mk = results.get('markitdown', {})
npm = results.get('npm', {})

# Aggregate by format
formats = {
    'PDF': {'mk_keys': ['pdf_slides'], 'npm_keys': ['pdf_slides'], 'npm_pkg': 'pdf-parse'},
    'DOCX': {'mk_keys': ['docx_narrative', 'docx_usecases', 'docx_retro'], 
             'npm_keys': ['docx_narrative', 'docx_usecases', 'docx_retro'], 'npm_pkg': 'mammoth'},
    'PPTX': {'mk_keys': ['pptx_diagram'], 'npm_keys': ['pptx_diagram'], 'npm_pkg': 'zip+xml'},
}

for fmt, info in formats.items():
    mk_total = sum(mk.get(k, {}).get('chars', 0) for k in info['mk_keys'])
    npm_total = sum(npm.get(k, {}).get('chars', 0) for k in info['npm_keys'])
    mk_errors = any(mk.get(k, {}).get('error') for k in info['mk_keys'])
    npm_errors = any(npm.get(k, {}).get('error') for k in info['npm_keys'])
    
    mk_status = f'{mk_total:,} chars' if not mk_errors else 'ERROR'
    npm_status = f'{npm_total:,} chars' if not npm_errors else 'ERROR/N/A'
    
    if mk_errors and npm_errors:
        winner = 'NEITHER'
    elif mk_errors:
        winner = info['npm_pkg']
    elif npm_errors:
        winner = 'markitdown'
    elif mk_total > npm_total * 1.2:
        winner = 'markitdown'
    elif npm_total > mk_total * 1.2:
        winner = info['npm_pkg']
    else:
        winner = 'TIE'
    
    print(f'| {fmt:<6} | {mk_status:<9} | {npm_status:<11} | {winner:<6} |       |')

print('| XLSX   | (not tested) | (not tested) | defer  | No files in dataset |')
print()
print('RECOMMENDATION:')
print('  - If markitdown wins all: use Python subprocess from preprocess-sources.sh')
print('  - If npm wins some: hybrid approach')
print('  - If both fail on visual content: confirms Phase 3 (vision) is necessary')

DECISION MATRIX — Fill after reviewing results above

| Format | markitdown | npm         | Winner | Notes |
|--------|-----------|-------------|--------|-------|
| PDF    | 31,310 chars | ERROR/N/A   | markitdown |       |
| DOCX   | 35,021 chars | 33,811 chars | TIE    |       |
| PPTX   | 1,305 chars | ERROR/N/A   | markitdown |       |
| XLSX   | (not tested) | (not tested) | defer  | No files in dataset |

RECOMMENDATION:
  - If markitdown wins all: use Python subprocess from preprocess-sources.sh
  - If npm wins some: hybrid approach
  - If both fail on visual content: confirms Phase 3 (vision) is necessary
